### DNA Sequence Embedding Calculation

#### 1. Function of this notebook
This notebook demonstrates how to calculate embeddings for DNA sequences using the pretrained PlasmidGPT model. 
The model encodes DNA sequences into fixed-length vector representations that capture sequence features.

#### 2. Input and Output
- **Input**: A DNA sequence string containing valid nucleotides (A, T, C, G, N)
- **Output**: A 1D numpy array of embedding values (mean-pooled hidden states from the last transformer layer)

#### 3. Time estimation
- Model loading: ~10-30 seconds (depending on hardware)
- Embedding calculation per sequence: ~1-2 seconds on GPU, ~5-10 seconds on CPU
- Total runtime for this demo: ~1 minute

In [2]:
import torch
from transformers import PreTrainedTokenizerFast
import numpy as np
import os

import torch
from transformers import PreTrainedTokenizerFast
import numpy as np
import os

# Function to check if a sequence contains only valid DNA characters (A, T, C, G, N)
def validate_sequence(sequence):
    valid_characters = set("ATCGN")
    sequence_upper = sequence.upper()
    if not set(sequence_upper).issubset(valid_characters):
        raise ValueError(f"Invalid character(s) found in sequence: {sequence}")
    return sequence_upper

# Function to load the model
def load_model(model_path, device):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file '{model_path}' not found.")
    
    model = torch.load(model_path)
    model.config.output_hidden_states = True  # Enable output of hidden states
    model.eval()
    model = model.to(device)
    
    return model

# Function to load the tokenizer
def load_tokenizer(tokenizer_file):
    if not os.path.exists(tokenizer_file):
        raise FileNotFoundError(f"Tokenizer file '{tokenizer_file}' not found.")
    
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_file)
    return tokenizer

# Function to calculate embeddings from the DNA sequence
def calculate_embeddings(model, tokenizer, sequence, device):
    input_ids = tokenizer.encode(sequence.upper(), return_tensors='pt',truncation=True, max_length=2048).to(device)
    
    # Inference to obtain hidden states
    with torch.no_grad():
        outputs = model(input_ids)
        hidden_states = outputs.hidden_states[-1].cpu().numpy()  # Get the last hidden state
        hidden_states_mean = np.mean(hidden_states, axis=1).reshape(-1)  # Compute mean along axis 1
    
    return hidden_states_mean

# Set the device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available, using CPU.")

# Define path to model directory (modify this path as needed)
model_dir = ""

model_path = os.path.join(model_dir, 'pretrained_model.pt')
tokenizer_path = os.path.join(model_dir, 'addgene_trained_dna_tokenizer.json')

# Load model and tokenizer
model = load_model(model_path, device)
tokenizer = load_tokenizer(tokenizer_path)
print("Model and tokenizer loaded successfully.")

# Define a DNA sequence
seq = "GTTACACCCTGTGAGCCTGCATGGGATGGATGAC"

# Validate the sequence
seq = validate_sequence(seq)
print(f"Sequence length: {len(seq)} bp")

# Calculate embedding
embedding = calculate_embeddings(model, tokenizer, seq, device)

print(f"Embedding shape: {embedding.shape}")
print(f"Embedding values (first 10): {embedding[:10]}")

# Save embedding to file (optional)
output_file = "single_sequence_embedding.txt"
np.savetxt(output_file, embedding.reshape(1, -1), fmt='%.6f')
print(f"Embedding saved to {output_file}")

Using GPU: NVIDIA GeForce RTX 3090 Ti
Model and tokenizer loaded successfully.
Sequence length: 34 bp
Embedding shape: (768,)
Embedding values (first 10): [-0.02134511  0.05301796 -0.5179837   0.03775753  0.20552798  0.373275
  0.20892878 -0.1507017   0.19840771  0.9652969 ]
Embedding saved to single_sequence_embedding.txt
